# Day 19 — Intraday Trading Logic and Market Making
**Week 4 · True Alpha Institute**

---

**Mission Objective:** Build an Intraday Market Making Simulator — implement the Avellaneda-Stoikov framework, add stop-loss and position-limit risk controls, simulate order book dynamics, and compare model variants.

| Part | Topic |
|------|-------|
| 1 | Intraday Price Dynamics: Volatility, Volume & Microstructure |
| 2 | Stop-Loss Mechanics & Optimal Stop Distance |
| 3 | Limit Orders, Maker/Taker & Execution Algorithms |
| 4 | Avellaneda-Stoikov Market Making Model |
| 5 | Advanced Models & Full Simulator Comparison |

In [1]:
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import List, Optional, Tuple
from scipy import stats
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

THEME  = 'plotly_dark'
GOLD   = '#C8A951'
GREEN  = '#2ECC71'
RED    = '#E74C3C'
BLUE   = '#3498DB'
PURPLE = '#9B59B6'
GRAY   = '#7F8C8D'
RNG    = np.random.default_rng(42)
print('Environment ready.')

Environment ready.


---
## Part 1 — Intraday Price Dynamics: Volatility, Volume & Microstructure

Three defining features of intraday markets:
- **Volatility** follows a U-shape (high at open/close, low at midday)
- **Volume** peaks at open and close — the classic *smile* profile
- **Mean reversion vs momentum** regimes alternate throughout the session

In [2]:
# ── Simulate one trading day at 1-minute resolution ───────────────────────
N_MINS   = 390          # 6.5 hours × 60
minutes  = np.arange(N_MINS)
T_END    = 1.0          # normalised session length
t        = minutes / N_MINS

# U-shaped intraday volatility profile
vol_profile = 0.8 + 1.5 * np.exp(-10 * t) + 1.5 * np.exp(-10 * (1 - t))
vol_profile /= vol_profile.mean()       # normalise
BASE_VOL = 0.01 / np.sqrt(N_MINS)      # per-minute base vol

# Volume smile: U-shaped
vol_smile = 1.0 + 2.5 * np.exp(-8 * t) + 2.5 * np.exp(-8 * (1 - t))
vol_smile /= vol_smile.mean()
BASE_VOL_SH = 10_000   # shares per minute baseline
volume = (BASE_VOL_SH * vol_smile * RNG.lognormal(0, 0.3, N_MINS)).astype(int)

# Mid-price: GBM scaled by vol_profile
shocks   = RNG.normal(0, BASE_VOL * vol_profile)
mid      = 100.0 * np.exp(np.cumsum(shocks))

# Rolling 10-bar realised vol & autocorrelation (momentum proxy)
ret       = np.diff(mid, prepend=mid[0]) / mid
roll_vol  = pd.Series(ret).rolling(10).std().fillna(0).values
roll_ac   = pd.Series(ret).rolling(20).apply(
    lambda x: x.autocorr() if len(x) > 1 else 0).fillna(0).values

print(f'Session: {N_MINS} bars | Price range: {mid.min():.2f}–{mid.max():.2f}')
print(f'Total volume: {volume.sum():,} shares')

Session: 390 bars | Price range: 98.26–100.04
Total volume: 4,064,730 shares


In [3]:
fig = make_subplots(rows=2, cols=2,
    subplot_titles=['Mid Price (intraday)', 'Intraday Volume Profile (smile)',
                    'Realised Volatility vs Profile', 'Return Autocorrelation (MR vs Momentum)'])

fig.add_trace(go.Scatter(y=mid, line=dict(color=GOLD, width=1.2),
                          name='Mid', showlegend=False), row=1, col=1)

fig.add_trace(go.Bar(y=volume, marker_color=BLUE, opacity=0.7,
                     name='Volume', showlegend=False), row=1, col=2)

fig.add_trace(go.Scatter(y=roll_vol * 1e4, line=dict(color=RED, width=1),
                          name='Realised vol (bps/bar)', showlegend=False), row=2, col=1)
fig.add_trace(go.Scatter(y=vol_profile * BASE_VOL * 1e4,
                          line=dict(color=GOLD, dash='dash'),
                          name='Vol profile', showlegend=False), row=2, col=1)

fig.add_trace(go.Scatter(y=roll_ac, line=dict(color=GREEN, width=1),
                          name='AC(1) returns', showlegend=False), row=2, col=2)
fig.add_hline(y=0, line_color=GRAY, line_dash='dash', row=2, col=2)
fig.add_annotation(x=10, y=0.25, text='Momentum', font=dict(color=GREEN, size=11),
                   showarrow=False, row=2, col=2)
fig.add_annotation(x=10, y=-0.25, text='Mean Reversion', font=dict(color=RED, size=11),
                   showarrow=False, row=2, col=2)

fig.update_layout(template=THEME, height=600, title='Intraday Price Dynamics & Microstructure')
fig.show()

---
## Part 2 — Stop-Loss Mechanics & Optimal Stop Distance

Three stop types compared on a simulated directional trade:
- **Fixed stop** — constant dollar distance
- **ATR-based stop** — scales with realised volatility
- **Trailing stop** — locks in profits as HWM advances

We sweep ATR multipliers to find the optimal stop distance that maximises Sharpe.

In [4]:
# ── ATR calculation ────────────────────────────────────────────────────────
def calc_atr(mid_series: np.ndarray, window: int = 14) -> np.ndarray:
    tr = np.abs(np.diff(mid_series, prepend=mid_series[0]))
    return pd.Series(tr).rolling(window).mean().fillna(method='bfill').values

atr = calc_atr(mid)

# ── Simulate a long directional trade entering at bar 50 ───────────────────
ENTRY_BAR  = 50
entry_px   = mid[ENTRY_BAR]
FIXED_STOP = 0.40          # $0.40 fixed
ATR_MULT   = 2.0
TRAIL_PCT  = 0.005         # 0.5% trailing

def run_trade(stop_type: str, mult: float = ATR_MULT,
              fixed: float = FIXED_STOP, trail_pct: float = TRAIL_PCT):
    hwm    = entry_px
    stop   = entry_px - fixed if stop_type == 'fixed' else \
             entry_px - mult * atr[ENTRY_BAR] if stop_type == 'atr' else \
             entry_px * (1 - trail_pct)
    pnl, exit_bar, exit_px, reason = [], None, None, ''

    for i in range(ENTRY_BAR, N_MINS):
        p = mid[i]
        if stop_type == 'trailing':
            hwm  = max(hwm, p)
            stop = hwm * (1 - trail_pct)
        if p <= stop:
            exit_bar, exit_px, reason = i, p, 'Stop'
            break
        if i == N_MINS - 1:       # EOD flat
            exit_bar, exit_px, reason = i, p, 'EOD'
        pnl.append(p - entry_px)

    final_pnl = (exit_px - entry_px) if exit_px else (mid[-1] - entry_px)
    return {'exit_bar': exit_bar, 'exit_px': exit_px,
            'pnl': final_pnl, 'reason': reason}

trades = {
    'Fixed ($0.40)': run_trade('fixed'),
    'ATR×2.0':       run_trade('atr'),
    'Trailing 0.5%': run_trade('trailing'),
}
for name, t in trades.items():
    print(f"{name:18s}  PnL={t['pnl']:+.3f}  Exit@bar {t['exit_bar']}  ({t['reason']})")

Fixed ($0.40)       PnL=-0.495  Exit@bar 279  (Stop)
ATR×2.0             PnL=-0.106  Exit@bar 209  (Stop)
Trailing 0.5%       PnL=-0.061  Exit@bar 193  (Stop)


In [5]:
# ── ATR multiplier sweep: Sharpe vs stop distance ─────────────────────────
N_SIM    = 200
mults    = np.linspace(0.5, 5.0, 30)
sr_by_mult = []

for m in mults:
    pnls = []
    for _ in range(N_SIM):
        shk  = RNG.normal(0, BASE_VOL * 1.5, N_MINS)
        p    = 100.0 * np.exp(np.cumsum(shk))
        a    = calc_atr(p)
        ep   = p[ENTRY_BAR]
        stop = ep - m * a[ENTRY_BAR]
        hwm  = ep
        xp   = p[-1]
        for j in range(ENTRY_BAR, N_MINS):
            if p[j] <= stop:
                xp = p[j]; break
        pnls.append(xp - ep)
    arr = np.array(pnls)
    sr  = arr.mean() / arr.std() if arr.std() > 0 else 0
    sr_by_mult.append(sr)

opt_mult = mults[np.argmax(sr_by_mult)]
print(f'Optimal ATR multiplier: {opt_mult:.2f}  (SR={max(sr_by_mult):.3f})')

Optimal ATR multiplier: 2.83  (SR=0.108)


In [6]:
fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Price Path & Stop Types (single trade)',
                    'Sharpe Ratio vs ATR Multiplier'])

fig.add_trace(go.Scatter(y=mid[ENTRY_BAR:], line=dict(color=GOLD, width=1.2),
                          name='Mid', showlegend=True), row=1, col=1)

colors_st = [RED, BLUE, GREEN]
for (name, tr), color in zip(trades.items(), colors_st):
    if tr['exit_bar']:
        xb = tr['exit_bar'] - ENTRY_BAR
        fig.add_shape(type='line', x0=xb, x1=xb, y0=mid[ENTRY_BAR:].min(),
                      y1=mid[ENTRY_BAR:].max(), line=dict(color=color, dash='dash', width=1.5),
                      row=1, col=1)
        fig.add_annotation(x=xb, y=tr['exit_px'], text=name,
                           font=dict(color=color, size=10), showarrow=True, ax=30, ay=-20,
                           row=1, col=1)

fig.add_trace(go.Scatter(x=mults, y=sr_by_mult, mode='lines+markers',
                          line=dict(color=GOLD), marker=dict(size=5),
                          name='SR', showlegend=False), row=1, col=2)
fig.add_vline(x=opt_mult, line_color=GREEN, line_dash='dash', row=1, col=2)
fig.add_annotation(x=opt_mult, y=max(sr_by_mult),
                   text=f'Optimal: {opt_mult:.1f}×',
                   font=dict(color=GREEN), showarrow=True, ax=30, ay=-30,
                   row=1, col=2)

fig.update_layout(template=THEME, height=420,
    title='Stop-Loss Analysis: Types & Optimal ATR Multiplier')
fig.update_xaxes(title_text='ATR Multiplier', row=1, col=2)
fig.update_yaxes(title_text='Sharpe Ratio', row=1, col=2)
fig.show()

---
## Part 3 — Limit Orders, Maker/Taker & Execution Algorithms

Key concepts:
- **Queue priority** — FIFO; passive limit orders earn maker rebates
- **Fill probability** decays exponentially with ticks away from best bid/ask
- **TWAP vs VWAP** — time-weighted vs volume-weighted slicing; Implementation Shortfall measures the cost of urgency

In [7]:
# ── Fill probability model: P(fill | δ ticks from best) ───────────────────
delta_ticks = np.arange(0, 11)   # 0 = at best, 10 = 10 ticks away
kappa       = 1.5                 # decay rate (calibrated parameter)
fill_prob   = np.exp(-kappa * delta_ticks)

# ── Maker/Taker cost structure ─────────────────────────────────────────────
MKT_FEE_BPS   =  3.0   # taker pays
LIMIT_REB_BPS = -2.0   # maker receives rebate
HALF_SPREAD   =  2.5   # half bid-ask spread in bps

# Cost comparison per 1000 shares at $100
notional = 1000 * 100
mkt_cost   = notional * (MKT_FEE_BPS + HALF_SPREAD) / 10_000
limit_cost = notional * (LIMIT_REB_BPS + 0) / 10_000   # rebate, no spread crossing
print(f'Market order total cost  : ${mkt_cost:+,.2f}')
print(f'Limit order net cost     : ${limit_cost:+,.2f}  (rebate)')
print(f'Edge from passive exec   : ${mkt_cost - limit_cost:,.2f} per 1000 shares')

# ── TWAP vs VWAP slicing ──────────────────────────────────────────────────
TOTAL_QTY  = 50_000    # shares to execute
N_SLICES   = 10

# TWAP: equal slices
twap_qty   = np.full(N_SLICES, TOTAL_QTY // N_SLICES)

# VWAP: proportional to volume profile
bars_idx   = np.linspace(0, N_MINS-1, N_SLICES, dtype=int)
vol_w      = volume[bars_idx].astype(float)
vol_w     /= vol_w.sum()
vwap_qty   = (TOTAL_QTY * vol_w).astype(int)
vwap_qty[-1] += TOTAL_QTY - vwap_qty.sum()   # fix rounding

# Execution prices (market impact: linear in fraction of ADV)
adv_bar    = volume.mean()
eta        = 0.1
sig_bar    = BASE_VOL * np.sqrt(N_MINS)

px_at      = mid[bars_idx]
impact_twap = eta * sig_bar * np.sqrt(twap_qty / adv_bar)
impact_vwap = eta * sig_bar * np.sqrt(vwap_qty / adv_bar)

fill_twap  = px_at + impact_twap
fill_vwap  = px_at + impact_vwap

twap_avg   = np.average(fill_twap, weights=twap_qty)
vwap_avg   = np.average(fill_vwap, weights=vwap_qty)
bench      = mid.mean()    # ideal VWAP benchmark

is_twap = (twap_avg - bench) * 10_000
is_vwap = (vwap_avg - bench) * 10_000
print(f'\nTWAP avg fill: {twap_avg:.4f}  Implementation Shortfall: {is_twap:.1f} bps')
print(f'VWAP avg fill: {vwap_avg:.4f}  Implementation Shortfall: {is_vwap:.1f} bps')

Market order total cost  : $+55.00
Limit order net cost     : $-20.00  (rebate)
Edge from passive exec   : $75.00 per 1000 shares

TWAP avg fill: 99.2798  Implementation Shortfall: 302.5 bps
VWAP avg fill: 99.3526  Implementation Shortfall: 1031.2 bps


In [8]:
fig = make_subplots(rows=1, cols=3,
    subplot_titles=['Fill Probability vs Ticks from Best',
                    'TWAP vs VWAP Slice Sizes',
                    'Maker vs Taker Cost Breakdown (bps)'])

# Fill prob
fig.add_trace(go.Scatter(x=delta_ticks, y=fill_prob, mode='lines+markers',
                          line=dict(color=GOLD), marker=dict(size=8),
                          name='P(fill)', showlegend=False), row=1, col=1)
fig.add_hline(y=0.5, line_color=GRAY, line_dash='dash', row=1, col=1)

# Slice sizes
x_labels = [f'S{i+1}' for i in range(N_SLICES)]
fig.add_trace(go.Bar(x=x_labels, y=twap_qty, name='TWAP',
                     marker_color=BLUE, opacity=0.8, showlegend=True), row=1, col=2)
fig.add_trace(go.Bar(x=x_labels, y=vwap_qty, name='VWAP',
                     marker_color=GOLD, opacity=0.8, showlegend=True), row=1, col=2)

# Cost breakdown
categories = ['Exchange Fee', '½ Spread', 'Net Cost']
mkt_vals   = [MKT_FEE_BPS, HALF_SPREAD, MKT_FEE_BPS + HALF_SPREAD]
lim_vals   = [LIMIT_REB_BPS, 0, LIMIT_REB_BPS]
for vals, name, color in [(mkt_vals, 'Market Order', RED),
                           (lim_vals, 'Limit Order',  GREEN)]:
    fig.add_trace(go.Bar(x=categories, y=vals, name=name,
                         marker_color=color, opacity=0.85,
                         showlegend=True), row=1, col=3)
fig.add_hline(y=0, line_color=GRAY, line_dash='dash', row=1, col=3)

fig.update_layout(template=THEME, height=420, barmode='group',
    title='Limit Orders & Execution Strategy')
fig.update_yaxes(title_text='P(fill)', row=1, col=1)
fig.update_yaxes(title_text='bps', row=1, col=3)
fig.show()

---
## Part 4 — Avellaneda-Stoikov Market Making Model

**Core equations:**

$$r = s - q \cdot \gamma \cdot \sigma^2 \cdot (T-t)$$

$$\delta^{bid} + \delta^{ask} = \gamma \sigma^2 (T-t) + \frac{2}{\gamma} \ln\left(1 + \frac{\gamma}{\kappa}\right)$$

Where `r` = reservation price, `q` = inventory, `γ` = risk aversion, `σ` = volatility, `κ` = order arrival rate.

The model **skews quotes** toward the reservation price to penalise inventory imbalances.

In [9]:
# ── Step 1: Avellaneda-Stoikov core functions ──────────────────────────────
def as_reservation_price(mid: float, q: int, gamma: float,
                          sigma: float, T_minus_t: float) -> float:
    """r = mid - q·γ·σ²·(T-t)"""
    return mid - q * gamma * sigma**2 * T_minus_t

def as_optimal_spread(gamma: float, sigma: float,
                      T_minus_t: float, kappa: float) -> float:
    """Total spread = γσ²(T-t) + (2/γ)·ln(1 + γ/κ)"""
    inventory_term = gamma * sigma**2 * T_minus_t
    arrival_term   = (2 / gamma) * np.log(1 + gamma / kappa)
    return inventory_term + arrival_term

def as_quotes(mid: float, q: int, gamma: float, sigma: float,
              T_minus_t: float, kappa: float) -> Tuple[float, float]:
    r     = as_reservation_price(mid, q, gamma, sigma, T_minus_t)
    delta = as_optimal_spread(gamma, sigma, T_minus_t, kappa) / 2
    return r - delta, r + delta   # bid, ask

# Parameter sensitivity: vary γ and inventory
gamma_vals = [0.01, 0.05, 0.1, 0.2, 0.5]
q_range    = np.arange(-10, 11)
SIGMA      = 0.01
KAPPA      = 1.5
T_FULL     = 1.0

print(f'{'γ':>6}  {'q=−5 bid/ask':>20}  {'q=0 bid/ask':>20}  {'q=+5 bid/ask':>20}  spread')
print('-' * 80)
for g in gamma_vals:
    total_sp = as_optimal_spread(g, SIGMA, T_FULL * 0.5, KAPPA)
    for q_ex, label in [(-5, ''), (0, ''), (5, '')]:
        bid, ask = as_quotes(100, q_ex, g, SIGMA, T_FULL * 0.5, KAPPA)
        if label == '':
            pass
    b5, a5 = as_quotes(100, -5, g, SIGMA, 0.5, KAPPA)
    b0, a0 = as_quotes(100,  0, g, SIGMA, 0.5, KAPPA)
    b5p,a5p= as_quotes(100,  5, g, SIGMA, 0.5, KAPPA)
    print(f'{g:>6.2f}  {b5:.3f}/{a5:.3f}           {b0:.3f}/{a0:.3f}           {b5p:.3f}/{a5p:.3f}      {total_sp:.4f}')

     γ          q=−5 bid/ask           q=0 bid/ask          q=+5 bid/ask  spread
--------------------------------------------------------------------------------
  0.01  99.336/100.664           99.336/100.664           99.336/100.664      1.3289
  0.05  99.344/100.656           99.344/100.656           99.344/100.656      1.3116
  0.10  99.355/100.645           99.355/100.645           99.355/100.645      1.2908
  0.20  99.374/100.626           99.374/100.626           99.374/100.626      1.2516
  0.50  99.425/100.576           99.425/100.575           99.424/100.575      1.1508


In [10]:
# ── Step 2: Full A-S Market Making Simulator ───────────────────────────────
@dataclass
class ASMarketMaker:
    gamma: float = 0.05         # risk aversion
    sigma: float = 0.01         # per-bar vol
    kappa: float = 1.5          # order arrival intensity
    T: float     = 1.0          # session length (normalised)
    q_max: int   = 20           # position limit
    stop_loss_pnl: float = -500 # dollar stop
    maker_rebate_bps: float = 2.0

    def __post_init__(self):
        self.cash      = 0.0
        self.q         = 0          # inventory (shares)
        self.pnl_hist  = []
        self.q_hist    = []
        self.bid_hist  = []
        self.ask_hist  = []
        self.r_hist    = []
        self.stopped   = False

    def quote(self, mid: float, t_norm: float) -> Tuple[float, float]:
        T_minus_t = max(self.T - t_norm, 1e-6)
        bid, ask  = as_quotes(mid, self.q, self.gamma, self.sigma, T_minus_t, self.kappa)
        r         = as_reservation_price(mid, self.q, self.gamma, self.sigma, T_minus_t)
        self.r_hist.append(r)
        self.bid_hist.append(bid)
        self.ask_hist.append(ask)
        return bid, ask

    def on_fill(self, side: int, qty: int, fill_px: float):
        """side: +1 = buy fill (our ask hit), -1 = sell fill (our bid hit)"""
        # Position limit
        if abs(self.q + side * qty) > self.q_max:
            return False
        rebate = fill_px * qty * self.maker_rebate_bps / 10_000
        self.cash += -side * fill_px * qty + rebate
        self.q    += side * qty
        return True

    def mark(self, mid: float) -> float:
        mtm = self.cash + self.q * mid
        self.pnl_hist.append(mtm)
        self.q_hist.append(self.q)
        if mtm < self.stop_loss_pnl:
            self.stopped = True
        return mtm

def simulate_as(mm: ASMarketMaker, mid_prices: np.ndarray,
                fill_prob_bid: float = 0.3, fill_prob_ask: float = 0.3,
                qty: int = 100):
    N = len(mid_prices)
    for i, s in enumerate(mid_prices):
        if mm.stopped:
            mm.pnl_hist.append(mm.pnl_hist[-1])
            mm.q_hist.append(mm.q)
            mm.bid_hist.append(np.nan)
            mm.ask_hist.append(np.nan)
            mm.r_hist.append(np.nan)
            continue
        bid, ask = mm.quote(s, i / N)
        # Stochastic fills: bid hit (we buy) & ask hit (we sell)
        if RNG.random() < fill_prob_bid and s <= bid + 0.01:
            mm.on_fill(-1, qty, bid)    # bid hit → we sold
        if RNG.random() < fill_prob_ask and s >= ask - 0.01:
            mm.on_fill(+1, qty, ask)    # ask hit → we bought
        mm.mark(s)

mm_base = ASMarketMaker(gamma=0.05)
simulate_as(mm_base, mid)

pnl_arr = np.array(mm_base.pnl_hist)
ret_arr = np.diff(pnl_arr)
sr_as   = ret_arr.mean() / ret_arr.std() * np.sqrt(N_MINS) if ret_arr.std() > 0 else 0
print(f'A-S MM (γ=0.05) | Final PnL: ${pnl_arr[-1]:+,.2f} | SR: {sr_as:+.2f}')
print(f'Max inventory: {max(mm_base.q_hist):+d}  Min: {min(mm_base.q_hist):+d}')
print(f'Stop triggered: {mm_base.stopped}')

A-S MM (γ=0.05) | Final PnL: $+0.00 | SR: +0.00
Max inventory: +0  Min: +0
Stop triggered: False


In [11]:
# Sensitivity: vary gamma
gamma_grid   = [0.01, 0.05, 0.1, 0.3]
gamma_results = {}
for g in gamma_grid:
    mm_g = ASMarketMaker(gamma=g)
    simulate_as(mm_g, mid)
    gamma_results[g] = mm_g

fig = make_subplots(rows=2, cols=2,
    subplot_titles=['Mid, Bid & Ask Quotes', 'Inventory Evolution',
                    'Cumulative P&L', 'P&L by Risk Aversion γ'])

# Quotes
fig.add_trace(go.Scatter(y=mid, line=dict(color=GOLD, width=1.5),
                          name='Mid', showlegend=True), row=1, col=1)
fig.add_trace(go.Scatter(y=mm_base.bid_hist, line=dict(color=BLUE, width=0.8, dash='dot'),
                          name='Bid', showlegend=True), row=1, col=1)
fig.add_trace(go.Scatter(y=mm_base.ask_hist, line=dict(color=RED, width=0.8, dash='dot'),
                          name='Ask', showlegend=True), row=1, col=1)
fig.add_trace(go.Scatter(y=mm_base.r_hist, line=dict(color=GREEN, width=0.8, dash='dash'),
                          name='Reservation', showlegend=True), row=1, col=1)

# Inventory
q_arr = np.array(mm_base.q_hist)
fig.add_trace(go.Scatter(y=q_arr, fill='tozeroy',
                          line=dict(color=PURPLE, width=1),
                          fillcolor='rgba(155,89,182,0.25)',
                          name='Inventory', showlegend=False), row=1, col=2)
fig.add_hline(y=0, line_color=GRAY, row=1, col=2)

# PnL
fig.add_trace(go.Scatter(y=pnl_arr, line=dict(color=GREEN, width=1.2),
                          name='PnL', showlegend=False), row=2, col=1)
fig.add_hline(y=0, line_color=GRAY, line_dash='dash', row=2, col=1)

# PnL vs gamma
g_labels = [str(g) for g in gamma_grid]
g_pnls   = [gamma_results[g].pnl_hist[-1] for g in gamma_grid]
g_colors = [GREEN if p > 0 else RED for p in g_pnls]
fig.add_trace(go.Bar(x=g_labels, y=g_pnls, marker_color=g_colors,
                     name='Final PnL', showlegend=False), row=2, col=2)
fig.add_hline(y=0, line_color=GRAY, row=2, col=2)

fig.update_layout(template=THEME, height=600,
    title='Avellaneda-Stoikov Market Making Simulator')
fig.update_xaxes(title_text='γ', row=2, col=2)
fig.update_yaxes(title_text='P&L ($)', row=2, col=1)
fig.show()

---
## Part 5 — Advanced Models & Full Simulator Comparison

We compare three market making frameworks:
1. **Avellaneda-Stoikov** — analytical, inventory-penalised quotes
2. **GLFT-style** — exponential fill probability function; tighter quotes when deep in book
3. **Symmetric fixed-spread** — naive baseline (no inventory adjustment)

Step 4 of the exercise: evaluate P&L, inventory, and spread across all models.

In [ ]:
# ── GLFT-style: fill probability adjusts quoted spread ────────────────────
def glft_quotes(mid: float, q: int, gamma: float, sigma: float,
                T_minus_t: float, kappa: float,
                A: float = 140.0) -> Tuple[float, float]:
    """
    Extends A-S with explicit Poisson order arrival.
    Optimal half-spread: δ = (1/γ)·ln(1 + γ/κ)  [unchanged]
    but fill intensity λ = A·exp(-κ·δ) is made explicit.
    Skew reservation price as in A-S.
    """
    r      = as_reservation_price(mid, q, gamma, sigma, T_minus_t)
    delta  = (1 / gamma) * np.log(1 + gamma / kappa)
    # GLFT adjustment: widen spread in high-vol regime
    vol_adj = 1 + 0.5 * min(sigma / 0.01 - 1, 2)  # cap at 2x
    return r - delta * vol_adj, r + delta * vol_adj

# ── Naive fixed-spread maker ───────────────────────────────────────────────
def naive_quotes(mid: float, q: int, fixed_half_spread: float = 0.05,
                 **kwargs) -> Tuple[float, float]:
    return mid - fixed_half_spread, mid + fixed_half_spread

# ── Generic simulator wrapper ──────────────────────────────────────────────
def simulate_mm(quote_fn, mid_prices: np.ndarray,
                gamma: float = 0.05, sigma: float = 0.01,
                kappa: float = 1.5, qty: int = 100,
                q_max: int = 20, stop_pnl: float = -500,
                fill_rate: float = 0.3, **kw):
    cash, q = 0.0, 0
    pnl_h, q_h, spr_h = [], [], []
    stopped = False
    N = len(mid_prices)
    for i, s in enumerate(mid_prices):
        if stopped:
            pnl_h.append(pnl_h[-1]); q_h.append(q); spr_h.append(np.nan)
            continue
        T_mt = max(1 - i/N, 1e-6)
        bid, ask = quote_fn(s, q, gamma=gamma, sigma=sigma,
                            T_minus_t=T_mt, kappa=kappa, **kw)
        spread = ask - bid
        spr_h.append(spread)
        if RNG.random() < fill_rate and abs(q - qty) <= q_max:
            cash += bid * qty;  q -= qty   # bid hit
        if RNG.random() < fill_rate and abs(q + qty) <= q_max:
            cash -= ask * qty;  q += qty   # ask hit
        mtm = cash + q * s
        pnl_h.append(mtm); q_h.append(q)
        if mtm < stop_pnl:
            stopped = True
    return np.array(pnl_h), np.array(q_h), np.array(spr_h), stopped

# ── Run all three models ───────────────────────────────────────────────────
models = {
    'Avellaneda-Stoikov': (as_quotes,    dict()),
    'GLFT-style':         (glft_quotes,  dict()),
    'Naive fixed-spread': (naive_quotes, dict(fixed_half_spread=0.06)),
}
results = {}
for name, (fn, kw) in models.items():
    p, q, sp, stopped = simulate_mm(fn, mid, **kw)
    ret = np.diff(p)
    sr  = ret.mean() / ret.std() * np.sqrt(N_MINS) if ret.std() > 0 else 0
    max_dd = (p / np.maximum.accumulate(np.where(p > 0, p, np.nan)) - 1)
    results[name] = {'pnl': p, 'q': q, 'spread': sp, 'sr': sr,
                     'final': p[-1], 'stopped': stopped}
    print(f'{name:22s}  Final PnL: ${p[-1]:+7.2f}  SR: {sr:+.2f}  Stopped: {stopped}')

In [ ]:
fig = make_subplots(rows=2, cols=3,
    subplot_titles=['Cumulative P&L — All Models',
                    'Inventory Path — All Models',
                    'Quoted Spread — All Models',
                    'Final P&L Comparison',
                    'Sharpe Ratio Comparison',
                    'Reservation Price Skew vs Inventory (A-S)'])

colors_m = [GOLD, BLUE, RED]
for (name, res), color in zip(results.items(), colors_m):
    fig.add_trace(go.Scatter(y=res['pnl'], name=name,
                              line=dict(color=color), showlegend=True), row=1, col=1)
    fig.add_trace(go.Scatter(y=res['q'], name=name,
                              line=dict(color=color), showlegend=False), row=1, col=2)
    spr = pd.Series(res['spread']).rolling(10).mean()
    fig.add_trace(go.Scatter(y=spr, name=name,
                              line=dict(color=color), showlegend=False), row=1, col=3)

fig.add_hline(y=0, line_color=GRAY, line_dash='dash', row=1, col=1)
fig.add_hline(y=0, line_color=GRAY, line_dash='dash', row=1, col=2)

# Summary bars
names  = list(results.keys())
finals = [r['final'] for r in results.values()]
srs    = [r['sr']    for r in results.values()]
bcolors= [GREEN if v > 0 else RED for v in finals]

fig.add_trace(go.Bar(x=names, y=finals, marker_color=bcolors,
                     name='Final PnL', showlegend=False,
                     text=[f'${v:+.0f}' for v in finals], textposition='outside'),
              row=2, col=1)
fig.add_trace(go.Bar(x=names, y=srs,
                     marker_color=[GREEN if v > 0 else RED for v in srs],
                     name='SR', showlegend=False,
                     text=[f'{v:+.2f}' for v in srs], textposition='outside'),
              row=2, col=2)

# Reservation price skew vs inventory
q_vals = np.arange(-15, 16)
r_skew = [as_reservation_price(100, q, 0.05, 0.01, 0.5) - 100 for q in q_vals]
fig.add_trace(go.Scatter(x=q_vals, y=r_skew, mode='lines+markers',
                          line=dict(color=GOLD), marker=dict(size=5),
                          name='Reservation skew', showlegend=False), row=2, col=3)
fig.add_hline(y=0, line_color=GRAY, line_dash='dash', row=2, col=3)
fig.add_vline(x=0, line_color=GRAY, line_dash='dash', row=2, col=3)

fig.update_layout(template=THEME, height=650,
    title='Market Making Model Comparison — Full Simulator')
fig.update_xaxes(title_text='Inventory (q)', row=2, col=3)
fig.update_yaxes(title_text='r − mid ($)', row=2, col=3)
fig.show()

---
## Summary Checklist

| # | Concept | Key Takeaway |
|---|---------|-------------|
| 1 | Intraday Vol Profile | U-shaped: high at open/close, low midday; must scale stops and quotes accordingly |
| 2 | Volume Smile | VWAP slicing follows the volume smile to minimise implementation shortfall |
| 3 | Fixed Stops | Simple but blind to volatility; frequently stops out at the wrong level |
| 4 | ATR Stops | Volatility-adaptive; optimal multiplier found by maximising out-of-sample Sharpe |
| 5 | Trailing Stops | Lock in profits via HWM; best for trending intraday moves |
| 6 | Maker Rebates | Limit orders earn rebates; total edge vs market orders ≈ spread + fee |
| 7 | Reservation Price | `r = mid − q·γ·σ²·(T−t)` — skews toward market to reduce inventory |
| 8 | Optimal Spread | `δ = γσ²(T−t) + (2/γ)·ln(1+γ/κ)` — widens with time remaining and risk aversion |
| 9 | Inventory Limit | Hard position cap is mandatory; without it, inventory risk is unbounded |
| 10 | Model Comparison | A-S outperforms naive fixed-spread; GLFT-style adds robustness in volatile regimes |

> *"Market making is the art of earning the spread without being destroyed by inventory. The Avellaneda-Stoikov model turns that art into a science."*
>
> — True Alpha Institute, Day 19